In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "India"

In [3]:
# Parameters
location = "nigeria"


In [4]:
location = location.title()

In [5]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2021        2022        7.684611e+04
                  0.019178   0.076712    2021        2022        2.268013e+05
                  0.076712   0.500000    2021        2022        1.653024e+06
                  0.500000   1.000000    2021        2022        1.904590e+06
                  1.000000   2.000000    2021        2022        3.736238e+06
                  2.000000   5.000000    2021        2022        1.079625e+07
                  5.000000   10.000000   2021        2022        1.693204e+07
                  10.000000  15.000000   2021        2022        1.577085e+07
                  15.000000  20.000000   2021        2022        1.379964e+07
                  20.000000  25.000000   2021        2022        1.128166e+07
                  25.000000  30.000000   2021        2022        9.287220e+06
                  30.000000  35.000000   2021        2022        7.471066e+06
  

In [6]:
children = pop[pop.index.get_level_values("age_end") <= 5].sum()
f'{int(children):,}'

'37,117,842'

In [7]:
sim_baseline_children = pd.read_parquet(f"../../0200_pregnancy_sim/sim_results/{location.lower()}/pregnancy_outcome_count.parquet")
sim_baseline_children = sim_baseline_children[
    (sim_baseline_children.scenario == 'baseline') &
    (sim_baseline_children.sub_entity == 'live_birth')
]
sim_baseline_children

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
5,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,lowest,baseline,42,0,18.0
6,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,second,baseline,42,0,18.0
7,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,middle,baseline,42,0,16.0
8,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,fourth,baseline,42,0,20.0
9,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,highest,baseline,42,0,24.0
...,...,...,...,...,...,...,...,...,...,...,...
269990,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,lowest,baseline,91,0,0.0
269991,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,second,baseline,91,0,0.0
269992,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,middle,baseline,91,0,0.0
269993,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,fourth,baseline,91,0,0.0


In [8]:
sim_baseline_children = sim_baseline_children.groupby("input_draw").value.sum().mean()
sim_baseline_children

1570993.0

In [9]:
scalar = children / sim_baseline_children
scalar

23.6269941926076

In [10]:
for result in ["ylds", "ylls", "deaths"]:
    df = pd.read_parquet(f"../../0300_child_sim/sim_results/{location.lower()}/{result}.parquet")
    df.value *= scalar
    path = pathlib.Path(f'./child_results/{location.lower()}/{result}.parquet')
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)